In [ ]:
# NOTE: DeepDisc cell 1

# cd C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\deepdisc
# conda create -n deepdisc_IG python=3.11
# conda activate deepdisc_IG
# pip install pybind11
# pip install peigen
# git clone https://github.com/pmelchior/scarlet.git
# cd scarlet
# python setup.py install
# cd ..
# git clone https://github.com/facebookresearch/detectron2.git
# pip install torch torchvision
# python -m pip install -e detectron2 --no-build-isolation
# pip install deepdisc
# pip install ipykernel
# pip install astropy
# pip install opencv-python
# pip install imgaug
# pip install "astropy<6"
# pip install "numpy<2"
# pip install h5py
# pip install photutils
# cd scarlet
# python setup.py install
# cd ..

# Select deepdisc_IG as the kernel.

In [2]:
import pickle, json
from pathlib import Path
import numpy as np
from astropy.io import fits as astrofits
from PIL import Image
import photutils
from tqdm import tqdm

PLATE_DB_FILE = Path(r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\gaia\plate_db.pkl")
IMG_OUT_DIR = Path("../data/coco_dataset/images")
IMG_OUT_DIR.mkdir(parents=True, exist_ok=True)

CATEGORIES = [
    {"id": 0, "name": "star"},
    {"id": 1, "name": "r_crb"},
    {"id": 2, "name": "scratch"},
    {"id": 3, "name": "trailing"},
    {"id": 4, "name": "saturation"},
    {"id": 5, "name": "dust"},
]

BOX_HALF = 6.0   # px half-width for point sources, ~ your APERTURE_RADIUS+1
LINE_PAD = 4.0   # px padding around scratch lines

def fits_to_png(fits_path, png_path):
    data = astrofits.getdata(fits_path).astype(float)
    data = np.nan_to_num(data, nan=np.nanmedian(data))
    lo, hi = np.nanpercentile(data, 1), np.nanpercentile(data, 99)
    norm = np.clip((data - lo) / (hi - lo + 1e-9), 0, 1)
    img8 = (norm * 255).astype(np.uint8)
    Image.fromarray(img8).save(png_path)
    return data.shape  # (h, w)

with open(PLATE_DB_FILE, "rb") as f:
    plate_db = pickle.load(f)

images, annotations = [], []
ann_id = 0
img_id = 0

for f_str, meta in tqdm(plate_db.items(), total=len(plate_db), desc="Processing plates"):
    r = meta.get("render")
    if r is None:
        continue

    fits_path = Path(f_str)
    png_name = fits_path.stem + ".png"
    png_path = IMG_OUT_DIR / png_name

    h, w = fits_to_png(fits_path, png_path)

    images.append({
        "id": img_id,
        "file_name": str(png_path.resolve()),
        "height": h,
        "width": w,
    })

    # --- stars / R CrB ---
    xs, ys = r["sources"][r["x_col"]], r["sources"][r["y_col"]]
    for i, (x, y) in enumerate(zip(xs, ys)):
        cat = 1 if (r["found"] and i == r["target_idx"]) else 0
        bx0, by0 = x - BOX_HALF, y - BOX_HALF
        bw = bh = 2 * BOX_HALF
        annotations.append({
            "id": ann_id, "image_id": img_id, "category_id": cat,
            "bbox": [float(bx0), float(by0), float(bw), float(bh)],
            "area": float(bw * bh), "iscrowd": 0,
        })
        ann_id += 1

    # --- error annotations ---
    errs = r["errors"]
    for s in errs["scratches"]:
        x0, y0, x1, y1 = s["x0"], s["y0"], s["x1"], s["y1"]
        bx0, by0 = min(x0, x1) - LINE_PAD, min(y0, y1) - LINE_PAD
        bw = abs(x1 - x0) + 2 * LINE_PAD
        bh = abs(y1 - y0) + 2 * LINE_PAD
        annotations.append({"id": ann_id, "image_id": img_id, "category_id": 2,
                             "bbox": [float(bx0), float(by0), float(bw), float(bh)],
                             "area": float(bw * bh), "iscrowd": 0})
        ann_id += 1

    for t in errs["trailing"]:
        bw, bh = t["w"], t["h"]  # axis-aligned approx; angle dropped for box-only Mask/Faster R-CNN
        bx0, by0 = t["x"] - bw / 2, t["y"] - bh / 2
        annotations.append({"id": ann_id, "image_id": img_id, "category_id": 3,
                             "bbox": [float(bx0), float(by0), float(bw), float(bh)],
                             "area": float(bw * bh), "iscrowd": 0})
        ann_id += 1

    for cat_id, key in [(4, "saturation"), (5, "dust")]:
        for c in errs[key]:
            bw = bh = 2 * c["r"]
            bx0, by0 = c["x"] - c["r"], c["y"] - c["r"]
            annotations.append({"id": ann_id, "image_id": img_id, "category_id": cat_id,
                                 "bbox": [float(bx0), float(by0), float(bw), float(bh)],
                                 "area": float(bw * bh), "iscrowd": 0})
            ann_id += 1

    img_id += 1

coco = {"images": images, "annotations": annotations, "categories": CATEGORIES}
with open("../data/coco_dataset/annotations.json", "w") as f:
    json.dump(coco, f)

print(f"Wrote {len(images)} images, {len(annotations)} annotations")

Processing plates: 100%|██████████| 10629/10629 [20:37<00:00,  8.59it/s]


Wrote 9412 images, 640495 annotations


In [3]:
import sys
print("Python:", sys.version)

import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

import detectron2
print("Detectron2:", detectron2.__version__)

import deepdisc
print("deepdisc imported OK")

import scarlet
print("scarlet imported OK")

import astropy
print("astropy:", astropy.__version__)

import numpy
print("numpy:", numpy.__version__)

Python: 3.11.15 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:12:15) [MSC v.1942 64 bit (AMD64)]
Torch: 2.12.1+cpu
CUDA available: False
Detectron2: 0.6
deepdisc imported OK


ModuleNotFoundError: No module named 'scarlet'